# Runing a simple RAG using OGX and OLLAMA

## First: Create A Vector Store for our RAG

In [1]:
import requests

OGX = "http://localhost:8321/v1"
provider= "faiss" 

def get_or_create_vector_store(existing_id=None):
    if existing_id:
        return existing_id
    res = requests.post(f"{OGX}/vector_stores", json={})
    return res.json()["id"]

VECTOR_STORE_ID = get_or_create_vector_store(None)
# VECTOR_STORE_ID = get_or_create_vector_store("vs_758c5e67-77cd-40fa-9994-fe06460f9b23")

print(VECTOR_STORE_ID)

vs_e1e2150a-be3f-4f6f-af4e-0e399f716e74


In [2]:
verify_res = requests.get(f"{OGX}/vector_stores")

if verify_res.status_code == 200:
    data = verify_res.json()
    
    # 1. Handle cases where the response is a dictionary with a nested list (e.g., {"data": [...]})
    if isinstance(data, dict):
        stores_list = data.get("data", []) if "data" in data else [data]
    else:
        stores_list = data

    # 2. Safely find your active store
    active_store = None
    if isinstance(stores_list, list):
        active_store = next((s for s in stores_list if isinstance(s, dict) and s.get("id") == VECTOR_STORE_ID), None)

    if active_store:
        print("--- Active Vector Store Configuration ---")
        print(f"Store ID:      {active_store.get('id')}")
        print(f"Type/Backend:  {active_store.get('vector_store_type')}")
        print(f"Metadata:      {active_store.get('metadata')}")
    else:
        print(f"ℹ️ Could not find your specific ID in the list, here is the raw response structure:")
        print(data)
else:
    print(f"Failed to fetch stores list: {verify_res.status_code}")

--- Active Vector Store Configuration ---
Store ID:      vs_e1e2150a-be3f-4f6f-af4e-0e399f716e74
Type/Backend:  None
Metadata:      {'provider_id': 'faiss', 'provider_vector_store_id': 'vs_e1e2150a-be3f-4f6f-af4e-0e399f716e74', 'embedding_model': 'sentence-transformers/nomic-ai/nomic-embed-text-v1.5', 'embedding_dimension': '768'}


## Second: Upload Files

In [3]:
import requests
from IPython.display import Markdown, display

OGX = "http://localhost:8321/v1"

def upload_and_attach_file(file_path, vector_store_id, show_preview=False):
    # -------------------------
    # 1. Upload file
    # -------------------------
    with open(file_path, "rb") as f:
        res = requests.post(
            f"{OGX}/files",
            files={"file": f},
            data={"purpose": "assistants"}
        )

    res.raise_for_status()
    file_id = res.json()["id"]
    print(f"[UPLOAD] file_id = {file_id}")

    # -------------------------
    # 2. Attach to vector store
    # -------------------------
    attach = requests.post(
        f"{OGX}/vector_stores/{vector_store_id}/files",
        json={"file_id": file_id}
    )

    attach.raise_for_status()
    print(f"[ATTACH] status = {attach.status_code}")

    # -------------------------
    # 3. Optional preview
    # -------------------------
    if show_preview:
        with open(file_path, "r", encoding="utf-8") as f:
            display(Markdown(f.read()))

    return {
        "file_id": file_id,
        "vector_store_id": vector_store_id,
        "attach_response": attach.json()
    }

Upload our files to the Vector store...

In [4]:
print("Upload to ... ",VECTOR_STORE_ID)
files = [
    "kubernetes_doc.txt",
    "ogx-ollama_doc.txt",
    "my-resume_doc.txt"
]

for file_path in files:
    result = upload_and_attach_file(
        file_path=file_path,
        vector_store_id=VECTOR_STORE_ID,
        show_preview=True
    )
    print("File ID:", result["file_id"])
    # print("Vector Store ID:", result["vector_store_id"])
    # print("Attach Response:", result["attach_response"])
    print("----------------------")

Upload to ...  vs_e1e2150a-be3f-4f6f-af4e-0e399f716e74
[UPLOAD] file_id = file-01822866765f4c8ca85c1252a156cf9f
[ATTACH] status = 200



Kubernetes Overview

Kubernetes is an open-source container orchestration platform used first by Oransa.
It automates deployment, scaling, and management of containerized applications.

Key concepts:
- Pods: the smallest deployable units
- Services: stable networking for pods
- Deployments: manage updates and replicas



File ID: file-01822866765f4c8ca85c1252a156cf9f
----------------------
[UPLOAD] file_id = file-b7a737d0a9e8406abaefb0fed1afd757
[ATTACH] status = 200



OGX is an AI gateway that exposes OpenAI-compatible APIs.
It can connect to Ollama models and provide file search capabilities.

Ollama is a local model runner that executes LLMs like Llama and DeepSeek.


File ID: file-b7a737d0a9e8406abaefb0fed1afd757
----------------------
[UPLOAD] file_id = file-adbdd9e0be544a7e98fd5ad836bd9424
[ATTACH] status = 200



Osama Oransa
SSA (Specialist Solution Architect) in MENA region at Red Hat
Based on Dubai, He enjoy playing tennis.


File ID: file-adbdd9e0be544a7e98fd5ad836bd9424
----------------------


## Third: Check if our files has be processed ...

In [5]:
import requests

OGX = "http://localhost:8321/v1"

res = requests.get(f"{OGX}/vector_stores/{VECTOR_STORE_ID}")
print("Upload to ... ",VECTOR_STORE_ID)
print(res.json()["file_counts"])

Upload to ...  vs_e1e2150a-be3f-4f6f-af4e-0e399f716e74
{'completed': 3, 'cancelled': 0, 'failed': 0, 'in_progress': 0, 'total': 3}


## Fourth: Call OGX to get the data:

Some helping formatting functions ...

In [6]:
# You can change the model name or relevant threshold
model = "ollama/llama3.2:3b"
relevant_threshold = 0.002

In [7]:
def full_response_output(resp):
    try:
        return resp["output"][0]["content"][0]["text"]
    except:
        return str(resp)
    
def format_response_output(resp_json):
    output = {
        "queries": [],
        "results": []
    }

    for item in resp_json.get("output", []):
        # tool call (retrieval)
        if item.get("type") == "file_search_call":
            output["queries"] = item.get("queries", [])

            for r in item.get("results", []):
                output["results"].append({
                    "text": r.get("text"),
                    "score": r.get("score"),
                    "probability": round((1 - r.get("score", 0)) * 100, 2)
                })

        # final answer
        if item.get("type") == "message":
            output["answer"] = item["content"][0]["text"]

    return output

def extract_specific_output(resp_json):
    try:
        for item in resp_json["output"]:
            if item.get("type") == "message":
                return item["content"][0]["text"]
    except Exception:
        pass

    return None
    
def get_top_score(resp_json):
    top_score = 0.0

    for item in resp_json.get("output", []):
        if item.get("type") == "file_search_call":
            for r in item.get("results", []):
                score = r.get("score", 0.0)
                top_score = max(top_score, score)

    return top_score
def not_relevant(top_score):
    if top_score < relevant_threshold:
        print("Didn't find any relevant answer...")
        return True
    else:
        print("Found relevant answer...")
        return False
    
def normal_llm(query):
    print("Revert back to model search...")
    r = requests.post(
        f"{OGX}/responses",
        json={
            "model": model,
            "temperature": 0.0,
            "input": query
        }
    )

    return r.json()["output"][0]["content"][0]["text"]

OGX RAG Search ...

In [8]:
def file_search(query, full_response=False):
    r = requests.post(
        f"{OGX}/responses",
        json={
            "model": model,
            "temperature": 0.0,
            "input": query,
            "tool_choice": "auto",
            "tools": [
                {
                    "type": "file_search",
                    "vector_store_ids": [VECTOR_STORE_ID],
                    "max_num_results": 1
                }
            ]
        }
    )
    resp_json = r.json()
    # print("return:", resp_json)
    top_score = get_top_score(resp_json)
    print("top_score=", top_score)
    if not_relevant(top_score):
        # revert back to normal LLM query
        return normal_llm(query)
        
    if full_response:
        return extract_specific_output(resp_json)

    return extract_specific_output(resp_json)

## Finally: Some test cases for our RAG

In [9]:
print(file_search("What does the document say about Kubernetes?", True))

top_score= 0.0048782354150541975
Found relevant answer...
Kubernetes is an open-source container orchestration platform used to automate deployment, scaling, and management of containerized applications. It was first developed by Oransa. Key concepts in Kubernetes include pods, which are the smallest deployable units, and services, which provide stable networking for these pods. Deployments are also a crucial aspect of Kubernetes, allowing users to manage updates and replicas of their applications.

This information is based on the search results provided earlier, specifically the document titled "kubernetes_doc.txt".


In [10]:
print(file_search("What does the document say about OGX?"))

top_score= 0.003523800818363166
Found relevant answer...
OGX is an AI gateway that exposes OpenAI-compatible APIs. It can connect to Ollama models and provide file search capabilities. Ollama is a local model runner that executes LLMs like Llama and DeepSeek.


In [11]:
print(file_search("What does the document say about Osama Oransa?",False))

top_score= 0.0044302822391826474
Found relevant answer...
I couldn't find any information on an individual named Osama Oransa. However, I found a document that mentions someone with a similar name, Osama Orans, who is an SSA (Specialist Solution Architect) in the MENA region at Red Hat. According to his profile, he is based in Dubai and enjoys playing tennis.


In [12]:
print(file_search("What does the document say about global warming?", True))

top_score= 0.0017191704079399146
Didn't find any relevant answer...
Revert back to model search...
I don't have any information about a specific document. If you could provide more context or details about the document, I'd be happy to try and help you understand what it says about global warming.

If you're looking for general information on global warming, I can provide some information. Global warming refers to the long-term rise in the overall temperature of the Earth's atmosphere, primarily caused by human activities such as burning fossil fuels and deforestation.

The Intergovernmental Panel on Climate Change (IPCC) is a leading international organization that provides scientific advice on climate change. According to the IPCC, global warming is real, and it's primarily caused by human activities that release greenhouse gases, such as carbon dioxide and methane, into the atmosphere.

The document may discuss various aspects of global warming, including:

1. Causes: Human activiti

In [13]:
print(file_search("What does the document say about Osa Ora?",False))

top_score= 0.002887603335936183
Found relevant answer...
I couldn't find any information on "Osa Ora". However, I found a mention of "Osama Oransa" which is an SSA (Specialist Solution Architect) in the MENA region at Red Hat based in Dubai. He enjoys playing tennis.


In [14]:
print(file_search("who is Red Hat?",False))

top_score= 0.002248968376352998
Found relevant answer...
Red Hat is an American multinational software company that specializes in open-source software, cloud computing, and enterprise technology. The company was founded in 1993 by Bob Young and Larry Augustin, and its headquarters are located in Raleigh, North Carolina.

Red Hat is best known for its Red Hat Enterprise Linux (RHEL) operating system, as well as its JBoss application server and OpenShift container platform. The company's products and services are used by a wide range of organizations, from small businesses to large enterprises, across various industries.

One of the key aspects of Red Hat's business model is its focus on open-source software. The company provides support and maintenance for its open-source products, as well as commercial versions with additional features and support. This approach has helped Red Hat build a large and loyal customer base, as well as a significant revenue stream from subscription-based se